# 🧬 LifeOps: GRPO Training — Chaotic Life Management

Trains `Qwen2.5-3B-Instruct` with GRPO to balance Career, Family, and Health.

Run this notebook on **Colab T4 (free tier)**.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade "trl>=0.15.0" transformers datasets httpx matplotlib openenv-core python-dotenv mergekit

## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("✅ Model loaded")

## 3. Launch LifeOps Server (Robust Background)

In [ ]:
import os
import subprocess
import time
import httpx
import sys

# 1. Extract and Navigate
if os.path.exists("LifeOps.zip"):
    !unzip -o LifeOps.zip
    # Check if unzipped into a subfolder
    if os.path.exists("LifeOps") and os.path.isdir("LifeOps"):
        %cd LifeOps

# 2. Launch Server with Logging
print("🚀 Launching LifeOps Server...")
with open("server_log.txt", "w") as log:
    subprocess.Popen(["python", "-m", "server.app"], 
                     stdout=log, stderr=log, 
                     env=dict(os.environ, PYTHONPATH=os.getcwd()))

# 3. Retry loop for connection
ENV_URL = "http://localhost:8000"
connected = False
for i in range(10):
    time.sleep(3)
    try:
        with httpx.Client(base_url=ENV_URL, timeout=5) as http:
            health = http.get("/health")
            if health.status_code == 200:
                print(f"✅ LifeOps Server is LIVE at {ENV_URL}")
                connected = True
                break
    except:
        print(f"Attempt {i+1}: Waiting for server...")

if not connected:
    print("❌ Server failed. LOG OUTPUT:")
    if os.path.exists("server_log.txt"):
        with open("server_log.txt", "r") as f: print(f.read())

## 4. Define Reward Function

In [ ]:
import re


def _extract_action_name(text: str) -> str | None:
    match = re.search(r"Action\s*:\s*([a-zA-Z_ -]+)", text, re.IGNORECASE)
    if not match:
        return None
    return re.sub(r"[\s-]+", "_", match.group(1).strip().lower())


def reward_fn(completions: list[str], **kwargs) -> list[float]:
    rewards = []
    with httpx.Client(base_url=ENV_URL, timeout=10) as http:
        for completion in completions:
            action_str = _extract_action_name(completion)
            if not action_str:
                rewards.append(-1.0)
                continue
            try:
                http.post("/reset")
                resp = http.post("/step", json={"action": {"choice": action_str, "justification": "RL Step"}})
                if resp.status_code == 200:
                    rewards.append(float(resp.json().get("reward", 0.0)))
                else:
                    rewards.append(-2.0)
            except Exception:
                rewards.append(0.0)
    return rewards

## 5. Prepare Dataset

In [ ]:
from datasets import Dataset
import sys
sys.path.append(os.getcwd())
from scripts.dataset_builder import LifeopsDatasetBuilder
dataset = Dataset.from_list([{"prompt": f"{item['prompt'][0]['content']}\n\n{item['prompt'][1]['content']}"} 
                             for item in LifeopsDatasetBuilder.generate_rl_dataset(100)])
print(f"✅ Generated {len(dataset)} examples")

## 6. GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Newer Unsloth versions already patch GRPO internals automatically.
# On Colab / Python 3.12, explicit PatchFastRL() can raise OSError.
try:
    from unsloth import PatchFastRL
    try:
        PatchFastRL()
    except OSError:
        pass
except Exception:
    pass

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=GRPOConfig(
        num_generations=4,
        max_steps=100,
        learning_rate=2e-5,
        max_prompt_length=384,
        max_completion_length=96,
        output_dir="./grpo_out",
        logging_steps=1,
        report_to="none",
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()